# BullsEye: A Multi-Stage Probing and Routing Architecture

Designed for **Google Colab Free Tier (T4 GPU)**.

### Quick-start for the demo only
If you just want to run the 5-image custom demo, jump straight to the **Custom Image Demo** section at the bottom. You only need:
1. Mount Drive
2. Clone repo + install deps
3. Pre-download model weights
4. Upload your images
5. Run custom demo cell

## ⚙️ Global Config — Set Once
Change `MODEL_ID` here to switch models across the entire notebook.

In [ ]:
# ── Model selection ───────────────────────────────────────────────────────
# 3B: faster download (~8 min), slightly lower accuracy
# 7B: slower download (~30 min), matches paper's reported numbers
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

# ── Output directory ─────────────────────────────────────────────────────
# Results are saved here. Switch to Drive path after mounting Drive below.
OUTPUT_DIR = "/content/drive/MyDrive/bullseye_results"

print(f"Model   : {MODEL_ID}")
print(f"Output  : {OUTPUT_DIR}")

## 💾 Mount Google Drive
Run this first so all results persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Drive mounted. Results will be saved to: {OUTPUT_DIR}")

## Setup: Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/tanzim12911/cse465-project.git"

import os
if not os.path.exists("cse465-project"):
    !git clone $REPO_URL
else:
    print("Repo already cloned, pulling latest...")
    !git -C cse465-project pull

os.chdir("cse465-project")
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Pre-download Model Weights

Run **once per session**. Downloads weights to Colab's local disk cache.
Every cell that loads the model afterwards reads from disk (~2-3 min) instead of the internet (~30 min).

> If your runtime resets, run this cell again before anything else.

In [ ]:
from huggingface_hub import snapshot_download
import time

print(f"Downloading weights for {MODEL_ID} ...")
start = time.time()
snapshot_download(
    repo_id=MODEL_ID,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"]
)
print(f"Done in {(time.time()-start)/60:.1f} min. Subsequent loads use local cache.")

## Download ColorBench Dataset

`--max_samples 500` streams only 500 rows — covers all 11 tasks without pulling the full 16 GB.
Remove the flag for the full dataset (not recommended on free tier).

In [ ]:
!python src/data/download_colorbench.py --max_samples 500

## Phase 0: Baseline Evaluation (Zero-Shot)
Runs the selected model on ColorBench with no interventions. Saves per-task accuracy to Drive.

In [ ]:
!python src/eval/baseline_eval.py \
    --model_id   $MODEL_ID \
    --output_dir $OUTPUT_DIR \
    --num_samples 500

## Phase 1: Blind Baseline
Strips images and runs text-only inference — shows how much performance comes from language priors alone.

In [ ]:
!python src/data/image_stripper.py

In [ ]:
!python src/eval/blind_eval.py \
    --model_id   $MODEL_ID \
    --output_dir $OUTPUT_DIR \
    --num_samples 500

## Phase 2: Three-Stage Probing
Extracts hidden states at vision encoder, mid-LLM, and final-LLM layers, then trains linear probes
to locate where color information degrades inside the model.

In [ ]:
!python src/probing/extract_features.py --model_id $MODEL_ID

In [ ]:
!python src/probing/train_probes.py

## Phase 3: Empirical Taxonomy
Uses probe accuracies to classify each task into one of three failure modes:
- **perception-limited** — signal never entered the model
- **prior-override** — signal existed early but was overridden by language priors
- **reasoning-soluble** — signal survived but model needed better prompting

Saves `taxonomy_map.json` which the dispatcher reads at runtime.

In [ ]:
!python src/probing/taxonomy_cluster.py

## Phase 4: BullsEye Evaluation
Routes each task to its matched intervention branch, then runs evaluation.
Results saved to Drive alongside baseline results for direct comparison.

In [ ]:
!python src/eval/bullseye_eval.py \
    --model_id   $MODEL_ID \
    --output_dir $OUTPUT_DIR \
    --num_samples 500

## Compare Baseline vs BullsEye Results
Reads both saved `.jsonl` files from Drive and prints a per-task comparison table.

In [ ]:
import json, os

def load_results(filepath):
    task_stats = {}
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return task_stats
    with open(filepath) as f:
        for line in f:
            d = json.loads(line)
            task = d.get('task', 'Unknown')
            task_stats.setdefault(task, {'correct': 0, 'total': 0})
            task_stats[task]['total'] += 1
            if d.get('correct'):
                task_stats[task]['correct'] += 1
    return task_stats

baseline = load_results(os.path.join(OUTPUT_DIR, 'baseline_results.jsonl'))
bullseye = load_results(os.path.join(OUTPUT_DIR, 'bullseye_results.jsonl'))

all_tasks = sorted(set(list(baseline.keys()) + list(bullseye.keys())))

print(f"{'Task':<25} {'Baseline':>10} {'BullsEye':>10} {'Delta':>8}")
print("-" * 58)

b_total_c, b_total_t = 0, 0
be_total_c, be_total_t = 0, 0

for task in all_tasks:
    b  = baseline.get(task, {'correct': 0, 'total': 0})
    be = bullseye.get(task, {'correct': 0, 'total': 0})
    b_acc  = b['correct']  / b['total']  * 100 if b['total']  > 0 else 0
    be_acc = be['correct'] / be['total'] * 100 if be['total'] > 0 else 0
    delta  = be_acc - b_acc
    sign   = '+' if delta >= 0 else ''
    print(f"{task:<25} {b_acc:>9.1f}% {be_acc:>9.1f}% {sign}{delta:>6.1f}%")
    b_total_c  += b['correct'];  b_total_t  += b['total']
    be_total_c += be['correct']; be_total_t += be['total']

print("-" * 58)
b_overall  = b_total_c  / b_total_t  * 100 if b_total_t  > 0 else 0
be_overall = be_total_c / be_total_t * 100 if be_total_t > 0 else 0
delta_overall = be_overall - b_overall
sign = '+' if delta_overall >= 0 else ''
print(f"{'OVERALL':<25} {b_overall:>9.1f}% {be_overall:>9.1f}% {sign}{delta_overall:>6.1f}%")

## Dataset Demo (auto-selects from ColorBench)
Streams 5 samples from ColorBench (one per task category) and runs both baseline and BullsEye side-by-side.
No full dataset download needed.

In [ ]:
!python src/eval/demo_examples.py \
    --model_id   $MODEL_ID \
    --output_dir $OUTPUT_DIR

## 📸 Custom Image Demo (Your Own 5 Images)

**Step 1** — Upload your 5 images using the cell below.

**Step 2** — Edit `CUSTOM_SAMPLES` in `src/eval/custom_demo.py` to set the question,
options, correct answer, and task name for each image.

**Step 3** — Run the demo cell. Results are printed and saved to Drive.

Task names that control which branch runs:
| Task | Branch applied |
|---|---|
| Color Counting, Color Comparison, Color Proportion, Object Counting, Color Blindness | Reasoning (CoT) |
| Color Extraction | Extraction (HSV inject) |
| Color Illusion, Color Mimicry | Suppression (grayscale) |
| Color Robustness | Normalization (gray world) |

In [ ]:
import os
from google.colab import files

os.makedirs('./demo_images', exist_ok=True)
print("Select your 5 images to upload...")
uploaded = files.upload()

for filename, data in uploaded.items():
    dest = f'./demo_images/{filename}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"Saved: {dest}")

print("\nUpdate CUSTOM_SAMPLES in src/eval/custom_demo.py to match these filenames.")

In [ ]:
!python src/eval/custom_demo.py \
    --model_id   $MODEL_ID \
    --output_dir $OUTPUT_DIR